# Supplemental alternate workflow — not the manuscript analysis

This notebook is retained as a user-supplied reference. It is not part of the canonical workflow described in the paper and has not been validated against the paper's methods or results. It uses a different Approach 1/2 assignment and legacy `$path` placeholders. Notebook execution outputs were removed from this project copy because they contain stale or discrepant results. The original files remain in the user's Downloads folder.


# scRNA-seq Pipeline — Step 2.1: Supervised Cell Re-Typing

> **Goal:** Starting from `scRNAseq_1_Preprocess.ipynb`'s CellTypist-annotated output, run HVG selection,
> a shared NB-VAE latent embedding, all 3 reannotation approaches (Approach 1 first — see background
> rationale below), merge everything into **one shared `adata`**, then cluster and evaluate. **UMAP is
> computed exactly once, at the very end** — not after every approach.

**Input:** `$path/5.pre_hvg_annotation/merged_celltypist_immune_low.h5ad`
(produced by `scRNAseq_1_Preprocess.ipynb`).

## Pipeline position

`scRNAseq_1_Preprocess.ipynb` → **`scRNAseq_2.1_CellReTypeSupervised.ipynb`** (this notebook) →
`scRNAseq_2.2_CellReTypeUnsupervised.ipynb` → `scRNAseq_3_GeneMarker.ipynb`

## Pipeline

| Step | What | Output |
|---|---|---|
| 1 | HVG selection (same as `process_pcluster_yg.ipynb` Step 6.1) | `2.1_suppervised/2.1.1_hvgs/` |
| 2 | NB-VAE latent embedding (`X_emb`) — shared by Approach 3 *and* clustering | `2.1_suppervised/2.1.2_nbvae/` |
| 3 | Approach 1 → Approach 2 → Approach 3 → consensus, **one shared `adata`** | `2.celltype_reannotate/` |
| 4 | kNN graph → Leiden clustering → batch entropy scoring (no UMAP yet) | `3.clustering/` |
| 5 | UMAP (once) + multi-panel evaluation of all 3 approaches + consensus + clusters | `3.clustering/` |

**Why Approach 1 first:** run Approach 1 (scANVI-style) as the primary call, Approach 2 (CellTypist
retrain) to extract markers, and take the consensus of all three to finalize
`adata.obs["consensus_celltype"]` — background rationale and the original 3-approach comparison are in
`cell_annotation_DL_v2_3.ipynb`.

**Why one shared `adata`:** each approach now writes its predictions directly onto the same in-memory
object instead of saving to 3 separate files and reconciling them afterward — the consensus step
becomes a simple vote, no file reconciliation needed.

Cell 2 | markdown
## Step 1 — Highly Variable Genes (HVGs)

Same method as `process_pcluster_yg.ipynb` Step 6.1 (batch-aware, top 3000 HVGs).

In [ ]:
# Cell 3 | code
import os
import numpy as np
import scanpy as sc

# ── Parameters ─────────────────────────────────────────────────────────────
n_hvgs    = 3000     # number of top HVGs (matches process_pcluster_yg.ipynb Step 6.1)
batch_key = "batch"  # batch-aware HVG selection across samples

# ── Paths ──────────────────────────────────────────────────────────────────
input_h5ad = "$path/5.pre_hvg_annotation/merged_celltypist_immune_low.h5ad"
hvgs_dir   = "$path/2.cell_reannotate/2.1_suppervised/2.1.1_hvgs"
os.makedirs(hvgs_dir, exist_ok=True)

prefix = "merged_hvgs"

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")
print(f"  Input  : {input_h5ad}")

# Preserve raw counts before log-normalisation
raw_counts = adata.X.copy()

# Log-normalise for HVG scoring
sc.pp.log1p(adata)

# Batch-aware HVG selection
kwargs = {"n_top_genes": n_hvgs}
if batch_key and batch_key in adata.obs.columns:
    kwargs["batch_key"] = batch_key

sc.pp.highly_variable_genes(adata, **kwargs)

# Save HVG boolean flags as pickle
adata.var[["highly_variable"]].to_pickle(os.path.join(hvgs_dir, f"{prefix}.pkl"))

# Restore raw counts
adata.X = raw_counts

# Save full-gene matrix (all genes, raw counts)
adata.write_h5ad(os.path.join(hvgs_dir, f"{prefix}_full.h5ad"))

# Subset to HVGs and save
n_total = adata.n_vars
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg.write_h5ad(os.path.join(hvgs_dir, f"{prefix}.h5ad"))

print(f"  HVGs selected : {adata_hvg.n_vars} / {n_total}")
print(f"  Output: {hvgs_dir}")
print("Step 1 — HVGs done.")

Cell 4 | markdown
## Step 2 — NB-VAE latent embedding (`X_emb`)

Same architecture/training loop as `process_pcluster_yg.ipynb` Step 6.2.1 (the method that notebook
chose as the best speed/performance trade-off). This embedding is used by **Approach 3** below *and*
by the **kNN/Leiden/entropy** clustering in Step 4 — computed once here, reused twice.

In [ ]:
# Cell 5 | code
import os
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.sparse import issparse
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# ── Parameters ─────────────────────────────────────────────────────────────
n_latent   = 30
n_hidden   = [256, 128]
n_epochs   = 800
batch_size = 128
lr         = 1e-4
kl_warmup  = 50       # epochs to linearly ramp KL weight from 0 → 1
patience   = 30       # early stopping patience (epochs)
val_frac   = 0.1      # fraction of cells for validation
batch_key  = "batch"  # obs column for batch correction (set "" to disable)

# ── Paths ──────────────────────────────────────────────────────────────────
input_h5ad = "$path/2.cell_reannotate/2.1_suppervised/2.1.1_hvgs/merged_hvgs.h5ad"
out_dir    = "$path/2.cell_reannotate/2.1_suppervised/2.1.2_nbvae"
os.makedirs(out_dir, exist_ok=True)
prefix = "merged_nbvae"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device : {device}")


# ── Model ──────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, n_input, n_hidden, n_latent, n_batch=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input + n_batch, n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
            nn.Linear(n_hidden[0], n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
        )
        self.fc_mu      = nn.Linear(n_hidden[1], n_latent)
        self.fc_log_var = nn.Linear(n_hidden[1], n_latent)

    def forward(self, x, batch_oh=None):
        inp = torch.cat([x, batch_oh], dim=1) if batch_oh is not None else x
        h = self.net(inp)
        return self.fc_mu(h), self.fc_log_var(h)


class Decoder(nn.Module):
    def __init__(self, n_latent, n_hidden, n_output, n_batch=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_latent + n_batch, n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
            nn.Linear(n_hidden[1], n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
        )
        self.fc_scale  = nn.Linear(n_hidden[0], n_output)
        # Per-gene log dispersion (shared across cells, learned during training)
        self.log_theta = nn.Parameter(torch.zeros(n_output))

    def forward(self, z, log_library, batch_oh=None):
        inp = torch.cat([z, batch_oh], dim=1) if batch_oh is not None else z
        h = self.net(inp)
        # Mean: softmax(scale) * library_size  → each gene's expected count
        px_scale = F.softmax(self.fc_scale(h), dim=-1)
        px_mu    = px_scale * torch.exp(log_library).unsqueeze(1)
        px_theta = torch.exp(self.log_theta)
        return px_mu, px_theta


class NBVAE(nn.Module):
    def __init__(self, n_genes, n_hidden, n_latent, n_batch=0):
        super().__init__()
        self.encoder = Encoder(n_genes, n_hidden, n_latent, n_batch)
        self.decoder = Decoder(n_latent, n_hidden, n_genes, n_batch)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std

    def forward(self, x, log_library, batch_oh=None):
        mu_z, log_var_z = self.encoder(torch.log1p(x), batch_oh)
        z = self.reparameterize(mu_z, log_var_z)
        px_mu, px_theta = self.decoder(z, log_library, batch_oh)
        return px_mu, px_theta, mu_z, log_var_z

    @torch.no_grad()
    def get_latent(self, x, log_library, batch_oh=None):
        self.eval()
        mu_z, _ = self.encoder(torch.log1p(x), batch_oh)
        return mu_z


# ── Loss ───────────────────────────────────────────────────────────────────
def nb_log_likelihood(x, mu, theta, eps=1e-8):
    log_theta_mu = torch.log(theta + mu + eps)
    return (
        theta * (torch.log(theta + eps) - log_theta_mu)
        + x   * (torch.log(mu    + eps) - log_theta_mu)
        + torch.lgamma(x + theta)
        - torch.lgamma(theta)
        - torch.lgamma(x + 1)
    )

def elbo(x, px_mu, px_theta, mu_z, log_var_z, kl_weight):
    recon = nb_log_likelihood(x, px_mu, px_theta).sum(dim=1).mean()
    kl    = -0.5 * (1 + log_var_z - mu_z.pow(2) - log_var_z.exp()).sum(dim=1).mean()
    return -recon + kl_weight * kl, recon.item(), kl.item()


# ── Data preparation ─────────────────────────────────────────────────────────
adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")

X = adata.X.toarray().astype(np.float32) if issparse(adata.X) else adata.X.astype(np.float32)
lib_size = X.sum(axis=1, keepdims=True).clip(1)
log_lib  = np.log(lib_size).squeeze(1)

# Batch one-hot encoding
n_batch_cats = 0
batch_oh_np  = None
if batch_key and batch_key in adata.obs.columns:
    cats = adata.obs[batch_key].astype("category")
    n_batch_cats = len(cats.cat.categories)
    batch_oh_np  = np.eye(n_batch_cats, dtype=np.float32)[cats.cat.codes.values]
    print(f"  Batches: {list(cats.cat.categories)}")

# Train / val split
n_cells  = X.shape[0]
n_val    = max(1, int(n_cells * val_frac))
idx      = np.random.permutation(n_cells)
val_idx, train_idx = idx[:n_val], idx[n_val:]

def make_tensors(indices):
    x_t   = torch.tensor(X[indices]).to(device)
    ll_t  = torch.tensor(log_lib[indices]).to(device)
    oh_t  = torch.tensor(batch_oh_np[indices]).to(device) if batch_oh_np is not None else None
    return TensorDataset(x_t, ll_t) if oh_t is None else TensorDataset(x_t, ll_t, oh_t)

train_loader = DataLoader(make_tensors(train_idx), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(make_tensors(val_idx),   batch_size=batch_size)

# ── Training ─────────────────────────────────────────────────────────────────
model = NBVAE(adata.n_vars, n_hidden, n_latent, n_batch_cats).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=lr)

train_losses, val_losses = [], []
best_val, no_improve, best_state = np.inf, 0, None

for epoch in range(1, n_epochs + 1):
    kl_w = min(1.0, epoch / kl_warmup)

    model.train()
    t_loss = 0
    for batch in train_loader:
        x_b, ll_b = batch[0], batch[1]
        oh_b = batch[2] if len(batch) == 3 else None
        px_mu, px_theta, mu_z, lv_z = model(x_b, ll_b, oh_b)
        loss, _, _ = elbo(x_b, px_mu, px_theta, mu_z, lv_z, kl_w)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        t_loss += loss.item() * x_b.size(0)
    train_losses.append(t_loss / len(train_idx))

    model.eval()
    v_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            x_b, ll_b = batch[0], batch[1]
            oh_b = batch[2] if len(batch) == 3 else None
            px_mu, px_theta, mu_z, lv_z = model(x_b, ll_b, oh_b)
            loss, _, _ = elbo(x_b, px_mu, px_theta, mu_z, lv_z, kl_weight=1.0)
            v_loss += loss.item() * x_b.size(0)
    val_losses.append(v_loss / len(val_idx))

    if val_losses[-1] < best_val:
        best_val, no_improve = val_losses[-1], 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1

    if epoch % 50 == 0:
        print(f"  Epoch {epoch:4d} | train {train_losses[-1]:.2f} | val {val_losses[-1]:.2f} | kl_w {kl_w:.2f}")

    if no_improve >= patience:
        print(f"  Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"  Best val loss: {best_val:.4f}")

# ── Loss curve ─────────────────────────────────────────────────────────────
plt.figure(figsize=(6, 3))
plt.plot(train_losses, label="train")
plt.plot(val_losses,   label="val")
plt.xlabel("Epoch"); plt.ylabel("ELBO loss"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{prefix}_loss.png"), dpi=150)
plt.show(); plt.close()

# ── Extract latent embedding ─────────────────────────────────────────────────
model.eval()
all_x   = torch.tensor(X).to(device)
all_ll  = torch.tensor(log_lib).to(device)
all_oh  = torch.tensor(batch_oh_np).to(device) if batch_oh_np is not None else None

with torch.no_grad():
    Z = model.get_latent(all_x, all_ll, all_oh).cpu().numpy()

adata.obsm["X_emb"] = Z

# ── Save ─────────────────────────────────────────────────────────────────────
adata.write_h5ad(os.path.join(out_dir, f"{prefix}.h5ad"))
pd.DataFrame(Z, index=adata.obs_names).to_pickle(os.path.join(out_dir, f"X_{prefix}.pkl"))
torch.save(best_state, os.path.join(out_dir, f"{prefix}_model.pt"))

print(f"  Latent shape : {Z.shape}")
print(f"  Output       : {out_dir}")
print("Step 2 — NB-VAE embedding done.")

Cell 6 | markdown
## Step 3 — Cell-type reannotation (3 approaches → one shared `adata`)

Order: **Approach 1 first** (primary call), **Approach 2 second** (marker extraction), **Approach 3
third** (uses the `X_emb` from Step 2), then **consensus**. All three write onto the *same* `adata`
loaded below — no UMAP is computed in this section.

In [ ]:
# Cell 7 | code
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# Same input used for Step 1 (full gene set, already CellTypist-annotated).
h5ad_path = "$path/5.pre_hvg_annotation/merged_celltypist_immune_low.h5ad"
adata = sc.read_h5ad(h5ad_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")

celltypist_label_cols = [c for c in adata.obs.columns if c.startswith("celltypist:") and not c.endswith(":conf")]
CELLTYPIST_COL = celltypist_label_cols[0]
CELLTYPIST_CONF_COL = CELLTYPIST_COL + ":conf"
CONF_THRESHOLD = 0.7

print(f"Using: {CELLTYPIST_COL}")
print(f"Confidence column: {CELLTYPIST_CONF_COL}")
print(f"Confidence threshold: {CONF_THRESHOLD}")

STAGE2_DIR = "$path/cell_reannotate/2.celltype_reannotate"
os.makedirs(STAGE2_DIR, exist_ok=True)

Cell 8 | markdown
### Approach 1 — Custom Semi-Supervised NB-VAE (scANVI-style), run first

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import issparse
from sklearn.model_selection import train_test_split

# ── Disable torch.compile to avoid circular imports ──────────────────
torch.compile = lambda x: x  # No-op: avoids loading torch._inductor

# ── Compatibility patch for broken torch._utils in notebook kernels ──────────
try:
    from torch import _utils
except (ImportError, AttributeError):
    print("Warning: torch._utils not available; attempting to patch...")
    import sys
    import importlib
    if 'torch' in sys.modules:
        try:
            importlib.reload(sys.modules['torch'])
        except Exception:
            pass
    try:
        from torch import _utils
        print("torch._utils recovered after reload")
    except (ImportError, AttributeError):
        class _UtilsStub:
            pass
        sys.modules['torch._utils'] = _UtilsStub()
        torch._utils = _UtilsStub()
        print("torch._utils stubbed; proceeding with training")

# ── Output directory for Approach 1 artifacts ─────────────────────────────────
OUTPUT_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.3_cell_reannotate/2.1.3.1_scANVI_nbvae"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cpu":
    torch.set_num_threads(os.cpu_count())
    print(f"CPU training: using all {os.cpu_count()} threads (torch.set_num_threads)")

# ── Step 1: load HVG raw-count object from Step 1 of this notebook ───────────
scvi_h5ad = "$path/2.cell_reannotate/2.1_suppervised/2.1.1_hvgs/merged_hvgs.h5ad"
adata_scvi = sc.read_h5ad(scvi_h5ad)
print(f"Loaded: {adata_scvi.n_obs} cells x {adata_scvi.n_vars} genes")

# ── Step 2: build semi-supervised labels (same logic as scANVI) ──────────────
label_values = adata.obs[CELLTYPIST_COL].reindex(adata_scvi.obs_names).values
label_values = np.array([str(v) for v in label_values], dtype=object)

conf_values = adata.obs[CELLTYPIST_CONF_COL].reindex(adata_scvi.obs_names).values
low_conf_mask = (pd.isna(conf_values)) | (conf_values < CONF_THRESHOLD)
label_values[low_conf_mask] = "Unknown"

adata_scvi.obs["train_labels"] = pd.Series(label_values, index=adata_scvi.obs_names, dtype=object)

n_labeled = (adata_scvi.obs["train_labels"] != "Unknown").sum()
n_unlabeled = (adata_scvi.obs["train_labels"] == "Unknown").sum()
print(f"Labeled cells   : {n_labeled}")
print(f'Unlabeled cells ("Unknown"): {n_unlabeled}')

# ── Step 3: prepare tensors ───────────────────────────────────────────────────
X = adata_scvi.X.toarray().astype(np.float32) if issparse(adata_scvi.X) else adata_scvi.X.astype(np.float32)
lib_size = X.sum(axis=1, keepdims=True).clip(1)
log_lib = np.log(lib_size).squeeze(1)

labels_str = adata_scvi.obs["train_labels"].values
labeled_mask = labels_str != "Unknown"

label_classes = sorted(pd.Series(labels_str[labeled_mask]).unique().tolist())
label_to_idx = {c: i for i, c in enumerate(label_classes)}
y = np.full(X.shape[0], -1, dtype=np.int64)
y[labeled_mask] = np.array([label_to_idx[v] for v in labels_str[labeled_mask]], dtype=np.int64)

labeled_idx = np.where(labeled_mask)[0]
n_classes = len(label_classes)

y_labeled = y[labeled_idx]
unique_labels, class_counts = np.unique(y_labeled, return_counts=True)
min_class_count = class_counts.min()
test_size = 0.1

min_needed = max(2, int(np.ceil(2 / test_size)))
can_stratify = (min_class_count >= min_needed)
if not can_stratify:
    print(f"Warning: stratification not feasible.")
    print(f"  Min class count: {min_class_count}")
    print(f"  Needed for stratified split (test_size={test_size}): {min_needed}")
    print("Using non-stratified split instead.")
    _, val_lab_idx = train_test_split(
        labeled_idx, test_size=min(test_size, 0.5), random_state=0, stratify=None,
    )
else:
    _, val_lab_idx = train_test_split(
        labeled_idx, test_size=test_size, random_state=0, stratify=y[labeled_idx],
    )

all_idx = np.arange(X.shape[0])
val_mask = np.zeros(X.shape[0], dtype=bool)
val_mask[val_lab_idx] = True
train_idx = all_idx[~val_mask]
val_idx = val_lab_idx

print(f"Train cells: {len(train_idx)}")
print(f"Val labeled cells: {len(val_idx)}")
print(f"Classes: {len(label_classes)}")

# ── Step 4: model (NB-VAE + classifier head) ─────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, n_input, n_hidden, n_latent):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
            nn.Linear(n_hidden[0], n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(n_hidden[1], n_latent)
        self.fc_log_var = nn.Linear(n_hidden[1], n_latent)

    def forward(self, x):
        h = self.net(torch.log1p(x))
        return self.fc_mu(h), self.fc_log_var(h)


class Decoder(nn.Module):
    def __init__(self, n_latent, n_hidden, n_output):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_latent, n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
            nn.Linear(n_hidden[1], n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
        )
        self.fc_scale = nn.Linear(n_hidden[0], n_output)
        self.log_theta = nn.Parameter(torch.zeros(n_output))

    def forward(self, z, log_library):
        h = self.net(z)
        px_scale = F.softmax(self.fc_scale(h), dim=-1)
        px_mu = px_scale * torch.exp(log_library).unsqueeze(1)
        px_theta = torch.exp(self.log_theta)
        return px_mu, px_theta


class SemiSupervisedNBVAE(nn.Module):
    def __init__(self, n_genes, n_hidden, n_latent, n_classes):
        super().__init__()
        self.encoder = Encoder(n_genes, n_hidden, n_latent)
        self.decoder = Decoder(n_latent, n_hidden, n_genes)
        self.classifier = nn.Sequential(
            nn.Linear(n_latent, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std

    def forward(self, x, log_library):
        mu_z, log_var_z = self.encoder(x)
        z = self.reparameterize(mu_z, log_var_z)
        px_mu, px_theta = self.decoder(z, log_library)
        cls_logits = self.classifier(mu_z)
        return px_mu, px_theta, mu_z, log_var_z, cls_logits

    @torch.no_grad()
    def get_latent(self, x):
        self.eval()
        mu_z, _ = self.encoder(x)
        return mu_z

    @torch.no_grad()
    def predict_labels(self, x):
        self.eval()
        mu_z, _ = self.encoder(x)
        logits = self.classifier(mu_z)
        probs = torch.softmax(logits, dim=1)
        return probs


def nb_log_likelihood(x, mu, theta, eps=1e-8):
    log_theta_mu = torch.log(theta + mu + eps)
    return (
        theta * (torch.log(theta + eps) - log_theta_mu)
        + x * (torch.log(mu + eps) - log_theta_mu)
        + torch.lgamma(x + theta)
        - torch.lgamma(theta)
        - torch.lgamma(x + 1)
    )


def loss_components(x, px_mu, px_theta, mu_z, log_var_z):
    recon = -nb_log_likelihood(x, px_mu, px_theta).sum(dim=1).mean()
    kl = -0.5 * (1 + log_var_z - mu_z.pow(2) - log_var_z.exp()).sum(dim=1).mean()
    return recon, kl


# ── Step 5: train (scANVI-style semi-supervised objective with NB-VAE) ───────
n_latent = 30
n_hidden = [256, 128]
n_epochs = 800
batch_size = 1024
lr = 1e-4
kl_warmup = 50
patience = 30
alpha_cls = 1.0

model = SemiSupervisedNBVAE(
    n_genes=adata_scvi.n_vars, n_hidden=n_hidden, n_latent=n_latent, n_classes=len(label_classes),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
ce_loss = nn.CrossEntropyLoss()

X_t = torch.tensor(X, dtype=torch.float32, device=device)
LL_t = torch.tensor(log_lib, dtype=torch.float32, device=device)
y_t = torch.tensor(y, dtype=torch.long, device=device)

val_idx_t = torch.tensor(val_idx, dtype=torch.long, device=device)

train_losses = []
val_losses = []
best_val = np.inf
no_improve = 0
best_state = None

for epoch in range(1, n_epochs + 1):
    kl_w = min(1.0, epoch / kl_warmup)
    model.train()

    perm = train_idx[np.random.permutation(len(train_idx))]
    epoch_loss = 0.0

    for start in range(0, len(perm), batch_size):
        bidx_np = perm[start:start + batch_size]
        bidx = torch.tensor(bidx_np, dtype=torch.long, device=device)

        x_b = X_t[bidx]
        ll_b = LL_t[bidx]
        y_b = y_t[bidx]

        px_mu, px_theta, mu_z, lv_z, logits = model(x_b, ll_b)
        recon, kl = loss_components(x_b, px_mu, px_theta, mu_z, lv_z)

        labeled_b = y_b >= 0
        if labeled_b.any():
            cls = ce_loss(logits[labeled_b], y_b[labeled_b])
        else:
            cls = torch.tensor(0.0, device=device)

        loss = recon + kl_w * kl + alpha_cls * cls

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        epoch_loss += loss.item() * len(bidx_np)

    train_losses.append(epoch_loss / len(train_idx))

    model.eval()
    with torch.no_grad():
        x_v = X_t[val_idx_t]
        ll_v = LL_t[val_idx_t]
        y_v = y_t[val_idx_t]

        px_mu_v, px_theta_v, mu_z_v, lv_z_v, logits_v = model(x_v, ll_v)
        recon_v, kl_v = loss_components(x_v, px_mu_v, px_theta_v, mu_z_v, lv_z_v)
        cls_v = ce_loss(logits_v, y_v)
        val_loss = (recon_v + kl_v + alpha_cls * cls_v).item()

    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1

    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | train={train_losses[-1]:.4f} | val={val_loss:.4f} | kl_w={kl_w:.2f}")

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"Best val loss: {best_val:.4f}")

# ── Step 5.1: plot training vs validation loss curve ────────────────────
fig_loss, ax_loss = plt.subplots(figsize=(8, 5))
ax_loss.plot(train_losses, label="train")
ax_loss.plot(val_losses, label="val")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss")
ax_loss.set_title("Approach 1 (NB-VAE) — training vs validation loss")
ax_loss.legend()
loss_curve_path = os.path.join(OUTPUT_DIR, "approach1_training_curve.png")
plt.savefig(loss_curve_path, dpi=150, bbox_inches="tight")
print(f"Training curve saved: {loss_curve_path}")
plt.show()

# ── Step 6: predictions + latent embedding (merged directly onto shared adata) ──
model.eval()
with torch.no_grad():
    Z = model.get_latent(X_t).cpu().numpy()
    probs = model.predict_labels(X_t).cpu().numpy()

pred_idx = probs.argmax(axis=1)
pred_conf = probs.max(axis=1)
pred_labels = np.array([label_classes[i] for i in pred_idx], dtype=object)

adata_scvi.obsm["X_scANVI"] = Z
adata_scvi.obs["approach1_celltype"] = pred_labels
adata_scvi.obs["approach1_conf"] = pred_conf

adata.obsm["X_scANVI"] = pd.DataFrame(Z, index=adata_scvi.obs_names).reindex(adata.obs_names).values
adata.obs["approach1_celltype"] = adata_scvi.obs["approach1_celltype"].reindex(adata.obs_names)
adata.obs["approach1_conf"] = adata_scvi.obs["approach1_conf"].reindex(adata.obs_names)

print("\nApproach 1 prediction distribution:")
print(adata.obs["approach1_celltype"].value_counts())

n_unknown_pred = (adata.obs["approach1_celltype"] == "Unknown").sum()
print(f"\nSanity check: 'Unknown' entries in approach1_celltype predictions: {n_unknown_pred}")

# ── Step 6.1: save predictions + model (no full-adata re-save; merged copy comes later) ──
pred_cols = ["approach1_celltype", "approach1_conf"]
pred_path = os.path.join(OUTPUT_DIR, "approach1_predictions.csv")
adata.obs[pred_cols].to_csv(pred_path)
print(f"Prediction table saved: {pred_path}")

model_path = os.path.join(OUTPUT_DIR, "scanvi_nbvae_model.pt")
torch.save(
    {
        "model_state": best_state,
        "label_classes": label_classes,
        "params": {
            "n_latent": n_latent, "n_hidden": n_hidden, "n_epochs": n_epochs,
            "batch_size": batch_size, "lr": lr, "kl_warmup": kl_warmup,
            "patience": patience, "alpha_cls": alpha_cls,
        },
    },
    model_path,
)
print(f"Model saved: {model_path}")

Cell 10 | markdown
### Approach 2 — Retrain CellTypist, run second (marker extraction)

Note: majority voting needs a Leiden cluster column, which doesn't exist yet at this point in the
pipeline (clustering happens in Step 4, after reannotation) — this cell will gracefully fall back to
per-cell annotation without majority voting, exactly as its existing fallback branch already handles.

In [ ]:
import inspect
import importlib
import contextlib
import celltypist
from celltypist import models as ct_models
from sklearn.linear_model import LogisticRegression as SkLogisticRegression

# ── Compatibility patch for scikit-learn >= 1.9 ─────────────────────────────
ct_train_module = importlib.import_module("celltypist.train")
if "multi_class" not in inspect.signature(SkLogisticRegression).parameters:
    def _logreg_compat(*args, multi_class=None, **kwargs):
        return SkLogisticRegression(*args, **kwargs)
    ct_train_module.LogisticRegression = _logreg_compat
    print("Applied CellTypist compatibility patch for scikit-learn >= 1.9")

# ── Output directory for Approach 2 artifacts ─────────────────────────────────
OUTPUT_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.3_cell_reannotate/2.1.3.2_celltypist_retrain"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Step 1: prepare normalized copy for CellTypist training ──────────────────
adata_train_ct = adata.copy()
sc.pp.normalize_per_cell(adata_train_ct, counts_per_cell_after=1e4)
sc.pp.log1p(adata_train_ct)

high_conf_mask = adata_train_ct.obs[CELLTYPIST_CONF_COL] > CONF_THRESHOLD
adata_train_ct = adata_train_ct[high_conf_mask].copy()
print(f"Training cells (conf > {CONF_THRESHOLD}): {adata_train_ct.n_obs} / {adata.n_obs}")
print(f"Cell types in training set: {adata_train_ct.obs[CELLTYPIST_COL].nunique()}")

# ── Step 2: train the dataset-specific model ────────────────────────────────
retrained_model = celltypist.train(
    adata_train_ct,
    labels=CELLTYPIST_COL,
    n_jobs=8,
    max_iter=200,
    feature_selection=True,
    top_genes=300,
    check_expression=False
)

RETRAINED_MODEL_PATH = os.path.join(OUTPUT_DIR, "retrained_celltypist.pkl")
retrained_model.write(RETRAINED_MODEL_PATH)
print(f"Model saved: {RETRAINED_MODEL_PATH}")

ANNOTATE_MODEL = retrained_model
print("Annotation model source: retrained model object (current run)")

@contextlib.contextmanager
def block_celltypist_downloads():
    original = getattr(ct_models, "download_models", None)

    def _blocked_download(*args, **kwargs):
        raise RuntimeError(
            "Blocked CellTypist built-in model download. "
            "Pass a valid in-memory model object or existing absolute .pkl path."
        )

    if original is not None:
        ct_models.download_models = _blocked_download
    try:
        yield
    finally:
        if original is not None:
            ct_models.download_models = original

# ── Step 3: inspect the learned dataset-specific markers ─────────────────────
coef_df = pd.DataFrame(
    retrained_model.classifier.coef_,
    index=retrained_model.cell_types,
    columns=retrained_model.features
)

print("\nTop 10 positive marker genes per cell type:")
marker_dict = {}
for ct in coef_df.index:
    top_markers = coef_df.loc[ct].nlargest(10).index.tolist()
    marker_dict[ct] = top_markers
    print(f"\n  {ct}:")
    print(f"    {top_markers}")

marker_table = pd.DataFrame(
    {ct: coef_df.loc[ct].nlargest(50).index.tolist() for ct in coef_df.index}
)
marker_path = os.path.join(OUTPUT_DIR, "retrained_celltypist_top50_markers.csv")
marker_table.to_csv(marker_path, index=False)
print(f"Top-marker table saved: {marker_path}")

# ── Step 4: re-annotate ALL cells with the selected model ─────────────────────
adata_full_ct = adata.copy()
sc.pp.normalize_per_cell(adata_full_ct, counts_per_cell_after=1e4)
sc.pp.log1p(adata_full_ct)

# Majority voting needs a cluster column — none exists yet (clustering is Step 4).
cluster_col = None
for c in ["leiden_0.5", "leiden_1.0", "leiden"]:
    if c in adata.obs.columns:
        cluster_col = c
        break

with block_celltypist_downloads():
    if cluster_col is None:
        print("No cluster column found yet (clustering happens in Step 4).")
        print("Running per-cell CellTypist annotation without majority voting.")
        pred_ct = celltypist.annotate(
            adata_full_ct,
            model=ANNOTATE_MODEL,
            majority_voting=False,
        )
    else:
        print(f"Using cluster column for majority voting: {cluster_col}")
        pred_ct = celltypist.annotate(
            adata_full_ct,
            model=ANNOTATE_MODEL,
            majority_voting=True,
            over_clustering=adata.obs[cluster_col],
        )

pred_ct_adata = pred_ct.to_adata()

if "majority_voting" in pred_ct_adata.obs.columns:
    adata.obs["approach2_celltype"] = pred_ct_adata.obs["majority_voting"]
else:
    adata.obs["approach2_celltype"] = pred_ct_adata.obs["predicted_labels"]

adata.obs["approach2_celltype_percell"] = pred_ct_adata.obs["predicted_labels"]
adata.obs["approach2_conf"] = pred_ct_adata.obs["conf_score"]

print("\nApproach 2 prediction distribution:")
print(adata.obs["approach2_celltype"].value_counts())

pred_cols = ["approach2_celltype", "approach2_celltype_percell", "approach2_conf"]
pred_path = os.path.join(OUTPUT_DIR, "approach2_predictions.csv")
adata.obs[pred_cols].to_csv(pred_path)
print(f"Prediction table saved: {pred_path}")

Cell 12 | markdown
### Approach 3 — Custom Neural Network on `X_emb`, run third

In [ ]:
# Cell 13 | code
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ── Output directory for Approach 3 artifacts ─────────────────────────────────
OUTPUT_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.3_cell_reannotate/2.1.3.3_customeNN"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Step 0: Load X_emb from this notebook's Step 2 NB-VAE embedding ───────────
adata_nbvae = sc.read_h5ad("$path/2.cell_reannotate/2.1_suppervised/2.1.2_nbvae/merged_nbvae.h5ad")
adata.obsm["X_emb"] = adata_nbvae.obsm["X_emb"]
print(f"X_emb loaded from Step 2 NB-VAE embedding: {adata.obsm['X_emb'].shape}")

# ── Step 1: prepare features and labels ───────────────────────────────────────
X_all = adata.obsm["X_emb"].astype(np.float32)   # (N_cells, 30) scVI latent embedding

mask = (adata.obs[CELLTYPIST_CONF_COL] > CONF_THRESHOLD).values

# fit label space on the high-confidence subset only, matching Approaches 1/2 —
# otherwise classes only ever seen with low confidence get an untrainable output slot
le = LabelEncoder()
y_conf = le.fit_transform(adata.obs[CELLTYPIST_COL][mask])
print(f"Cell types ({len(le.classes_)}): {le.classes_.tolist()}")
X_conf = X_all[mask]

try:
    X_train, X_val, y_train, y_val = train_test_split(
        X_conf, y_conf, test_size=0.2, stratify=y_conf, random_state=0
    )
except ValueError:
    print("Warning: stratification not feasible (some classes have <2 members).")
    print("Using non-stratified split instead.")
    X_train, X_val, y_train, y_val = train_test_split(
        X_conf, y_conf, test_size=0.2, stratify=None, random_state=0
    )
print(f"Training cells  : {len(X_train)}")
print(f"Validation cells: {len(X_val)}")

# ── Step 2: define the network ────────────────────────────────────────────────
class CellTypeNet(nn.Module):
    def __init__(self, n_input, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64),     nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)

n_classes = len(le.classes_)
model_nn  = CellTypeNet(n_input=30, n_classes=n_classes)
optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

X_tr = torch.tensor(X_train)
y_tr = torch.tensor(y_train)
X_vl = torch.tensor(X_val)
y_vl = torch.tensor(y_val)

# ── Step 3: training loop (mini-batch + early stopping, matching Approach 2) ──
n_epochs = 800
batch_size = 256
patience = 30

train_losses = []
val_losses   = []
val_accs     = []
best_val_loss = np.inf
no_improve = 0
best_state = None

n_train = len(X_tr)
for epoch in range(1, n_epochs + 1):
    model_nn.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0

    for start in range(0, n_train, batch_size):
        bidx = perm[start:start + batch_size]
        x_b, y_b = X_tr[bidx], y_tr[bidx]

        optimizer.zero_grad()
        loss = loss_fn(model_nn(x_b), y_b)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(bidx)

    train_losses.append(epoch_loss / n_train)

    model_nn.eval()
    with torch.no_grad():
        val_logits = model_nn(X_vl)
        val_loss = loss_fn(val_logits, y_vl).item()
        val_acc = (val_logits.argmax(1) == y_vl).float().mean().item()
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        best_state = {k: v.detach().clone() for k, v in model_nn.state_dict().items()}
    else:
        no_improve += 1

    if epoch % 25 == 0:
        print(f"  Epoch {epoch:4d}  train_loss={train_losses[-1]:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model_nn.load_state_dict(best_state)
print(f"Best val loss: {best_val_loss:.4f}")

# plot training curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="train")
axes[0].plot(val_losses, label="val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy loss"); axes[0].set_title("Training/validation loss"); axes[0].legend()
axes[1].plot(val_accs);     axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy");           axes[1].set_title("Validation accuracy")
plt.tight_layout()
training_curve_path = os.path.join(OUTPUT_DIR, "approach3_training_curve.png")
plt.savefig(training_curve_path, dpi=150, bbox_inches="tight")
print(f"Training curve saved: {training_curve_path}")
plt.show()

# ── Step 4: predict all cells ─────────────────────────────────────────────────
model_nn.eval()
with torch.no_grad():
    probs = torch.softmax(model_nn(torch.tensor(X_all)), dim=1).numpy()

adata.obs["approach3_celltype"] = le.inverse_transform(probs.argmax(axis=1))
adata.obs["approach3_conf"]     = probs.max(axis=1)

print("\nApproach 3 prediction distribution:")
print(adata.obs["approach3_celltype"].value_counts())

# ── Step 5: save predictions + model (no UMAP here; deferred to Step 5 of notebook) ──
pred_cols = ["approach3_celltype", "approach3_conf"]
pred_path = os.path.join(OUTPUT_DIR, "approach3_predictions.csv")
adata.obs[pred_cols].to_csv(pred_path)
print(f"Prediction table saved: {pred_path}")

model_path = os.path.join(OUTPUT_DIR, "customnn_model.pt")
torch.save({
    "model_state":         model_nn.state_dict(),
    "label_classes":       le.classes_.tolist(),
    "celltypist_col_used": CELLTYPIST_COL,
    "n_input":             30,
    "n_classes":           n_classes,
}, model_path)
print(f"Model saved: {model_path}")

Cell 14 | markdown
### Consensus — merge into `consensus_celltype` / `consensus_agreement`

Since all three approaches wrote onto the *same* `adata`, the consensus step is a straightforward
per-cell vote — no file reconciliation needed.

In [ ]:
# Cell 15 | code
from collections import Counter

def consensus_vote(row):
    votes = [row["approach1_celltype"], row["approach2_celltype"], row["approach3_celltype"]]
    most_common, count = Counter(votes).most_common(1)[0]
    if count == 3:
        return most_common, "all agree"
    elif count == 2:
        return most_common, "2/3 agree"
    return "review needed", "review needed"

votes_df = adata.obs.apply(consensus_vote, axis=1, result_type="expand")
adata.obs["consensus_celltype"]  = votes_df[0]
adata.obs["consensus_agreement"] = votes_df[1]

n_all_agree = (adata.obs["consensus_agreement"] == "all agree").sum()
n_majority  = (adata.obs["consensus_agreement"] == "2/3 agree").sum()
n_review    = (adata.obs["consensus_agreement"] == "review needed").sum()
print(f"\nConsensus (per cell, {adata.n_obs} total):")
print(f"  All 3 agree  : {n_all_agree}")
print(f"  2/3 agree    : {n_majority}")
print(f"  Review needed: {n_review}")

print("\nFinal consensus label distribution:")
print(adata.obs["consensus_celltype"].value_counts())

# ── Save the merged, all-3-approaches adata (one file, no reconciliation needed) ──
CONSENSUS_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.3_cell_reannotate/2.1.3.4_consensus"
os.makedirs(CONSENSUS_DIR, exist_ok=True)
final_h5ad_path = os.path.join(CONSENSUS_DIR, "adata_reannotated.h5ad")
adata.write_h5ad(final_h5ad_path)
print(f"\nSaved: {final_h5ad_path}")

Cell 16 | markdown
## Step 4 — kNN graph, Leiden clustering, batch entropy scoring

Same methods as `process_pcluster_yg.ipynb` Steps 7.2/7.4/7.5, built on the `X_emb` from Step 2
(the same embedding Approach 3 used). **No UMAP yet** — that's Step 5.

#### Step 4.1 KNN graph

In [ ]:
# Cell 17 | code
import os
import scanpy as sc
from threadpoolctl import threadpool_limits

# ── Parameters ─────────────────────────────────────────────────────────────
use_rep   = "X_emb"
n_pcs     = 30
n_threads = 4

# ── Paths ──────────────────────────────────────────────────────────────────
input_h5ad = "$path/2.cell_reannotate/2.1_suppervised/2.1.3_cell_reannotate/2.1.3.4_consensus/adata_reannotated.h5ad"
STAGE3_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.4_clustering/2.1.4.1_knn"
os.makedirs(STAGE3_DIR, exist_ok=True)

threadpool_limits(n_threads)
sc.settings.n_jobs = n_threads

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")
print(f"  X_emb  : {adata.obsm['X_emb'].shape}")

sc.pp.neighbors(adata, use_rep=use_rep, n_pcs=n_pcs)
print(f"  kNN graph built  (use_rep={use_rep}, n_pcs={n_pcs})")

adata.write_h5ad(os.path.join(STAGE3_DIR, "adata_knn.h5ad"))
print(f"  Saved  : {STAGE3_DIR}/adata_knn.h5ad")
print("Step 4a — kNN done.")

#### Step 4.2 clustering

In [ ]:
# Cell 18 | code
import os
import scanpy as sc

resolutions = [0.5, 1.0]

input_h5ad = "$path/2.cell_reannotate/2.1_suppervised/2.1.4_clustering/2.1.4.1_knn/adata_knn.h5ad"
STAGE3_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.4_clustering/2.1.4.2_cluster"
os.makedirs(STAGE3_DIR, exist_ok=True)

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells")

for res in resolutions:
    key = f"leiden_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key)
    n_clusters = adata.obs[key].nunique()
    print(f"  Resolution {res} \u2192 {n_clusters} clusters  (key: '{key}')")
    adata.obs[[key]].to_pickle(os.path.join(STAGE3_DIR, f"leiden_r{res}.pkl"))

# No per-resolution UMAP plot here (UMAP doesn't exist yet) — see Step 5.
adata.write_h5ad(os.path.join(STAGE3_DIR, "adata_leiden.h5ad"))
print(f"  Saved  : {STAGE3_DIR}/adata_leiden.h5ad")
print(f"  Cluster columns: {[f'leiden_{r}' for r in resolutions]}")
print("Step 4b — Leiden done.")

#### Step 4.3 Entropy

In [ ]:
# Cell 19 | code
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import entropy

# ── Parameters ─────────────────────────────────────────────────────────────
entropy_col = "batch"          # column holding batch labels
resolutions = [0.5, 1.0]      # must match Step 4b

# ── Paths ──────────────────────────────────────────────────────────────────
input_h5ad  = "$path/2.cell_reannotate/2.1_suppervised/2.1.4_clustering/2.1.4.2_cluster/adata_leiden.h5ad"
STAGE3_DIR  = "$path/2.cell_reannotate/2.1_suppervised/2.1.4_clustering/2.1.4.3_entropy"
os.makedirs(STAGE3_DIR, exist_ok=True)

adata = sc.read_h5ad(input_h5ad)
batches = adata.obs[entropy_col].unique().tolist()
K = len(batches)
print(f"  Loaded : {adata.n_obs} cells | batches (K={K}) : {batches}")

def cluster_entropy(obs_df, cluster_col, entropy_col, K):
    """Shannon entropy (normalized by log2(K)) per cluster."""
    records = []
    for cluster, grp in obs_df.groupby(cluster_col):
        counts = grp[entropy_col].value_counts()
        probs  = counts / counts.sum()
        H      = entropy(probs, base=2)
        H_norm = H / np.log2(K) if K > 1 else 0.0
        records.append({"cluster": cluster, "entropy": H, "entropy_norm": H_norm, "n_cells": len(grp)})
    return pd.DataFrame(records).sort_values("cluster")

all_entropy = {}
for res in resolutions:
    key = f"leiden_{res}"
    df  = cluster_entropy(adata.obs, key, entropy_col, K)
    all_entropy[res] = df
    print(f"\n  Resolution {res}:")
    print(df.to_string(index=False))

    df.to_pickle(os.path.join(STAGE3_DIR, f"entropy_r{res}.pkl"))
    df.to_csv(os.path.join(STAGE3_DIR, f"entropy_r{res}.csv"), index=False)

    fig, ax = plt.subplots(figsize=(max(6, len(df) * 0.5), 4))
    ax.bar(df["cluster"].astype(str), df["entropy_norm"], color="steelblue", edgecolor="white")
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8, label="max entropy")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Normalized Shannon Entropy")
    ax.set_title(f"Batch Entropy per Cluster (res={res})")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(STAGE3_DIR, f"entropy_r{res}.png"), dpi=150, bbox_inches="tight")
    plt.close()

# Attach per-cell entropy (from cluster membership) back to adata
for res in resolutions:
    key     = f"leiden_{res}"
    ent_map = all_entropy[res].set_index("cluster")["entropy_norm"]
    adata.obs[f"entropy_norm_{res}"] = adata.obs[key].map(ent_map)

adata.write_h5ad(os.path.join(STAGE3_DIR, "adata_entropy.h5ad"))
print(f"\n  Saved  : {STAGE3_DIR}/adata_entropy.h5ad")
print("Step 4c — Entropy done.")

Cell 20 | markdown
## Step 5 — UMAP (once) + multi-panel evaluation

This is the **only** UMAP computed in the entire notebook, and it's computed **once** from the same
kNN graph already built on `X_emb` in Step 4a — so the Leiden clusters and batch-entropy values plotted
below are geometrically coherent with this layout (they were derived from this same graph).

Panels: atlas input, Approach 1/2/3 predictions, consensus, Leiden clusters, and batch entropy —
all on one consistent 2D layout, so the 3 approaches can be visually compared directly.

In [ ]:
# Cell 21 | code
import os
import scanpy as sc
import matplotlib.pyplot as plt

ENTROPY_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.4_clustering/2.1.4.3_entropy"
input_h5ad = os.path.join(ENTROPY_DIR, "adata_entropy.h5ad")

STAGE3_DIR = "$path/2.cell_reannotate/2.1_suppervised/2.1.5_umap_evaluate"
os.makedirs(STAGE3_DIR, exist_ok=True)

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells")

# Resolve the atlas label column locally (don't depend on Step 3's in-memory state).
celltypist_label_cols = [c for c in adata.obs.columns if c.startswith("celltypist:") and not c.endswith(":conf")]
CELLTYPIST_COL = celltypist_label_cols[0]

# Reuses the kNN graph already built on X_emb in Step 4a.
sc.tl.umap(adata, random_state=0)
print(f"  UMAP shape : {adata.obsm['X_umap'].shape}")

adata.write_h5ad(os.path.join(STAGE3_DIR, "adata_umap.h5ad"))
print(f"  Saved  : {STAGE3_DIR}/adata_umap.h5ad")
print("Step 5a — UMAP done (the only UMAP in this notebook).")

In [ ]:
# Cell 22 | code
FINAL_OUT = os.path.join(STAGE3_DIR, "umap_evaluation.png")

# ── Shared color map so the same cell type gets the same color in every panel ──
import matplotlib as mpl

celltype_cols = [CELLTYPIST_COL, "approach1_celltype", "approach2_celltype", "approach3_celltype", "consensus_celltype"]
all_celltypes = sorted(set().union(*[adata.obs[c].astype(str).unique().tolist() for c in celltype_cols]))

palette_cycle = (
    list(mpl.colormaps["tab20"].colors)
    + list(mpl.colormaps["tab20b"].colors)
    + list(mpl.colormaps["tab20c"].colors)
)
celltype_color_map = {ct: mpl.colors.to_hex(palette_cycle[i % len(palette_cycle)]) for i, ct in enumerate(all_celltypes)}

for col in celltype_cols:
    adata.obs[col] = adata.obs[col].astype(str).astype("category")
    adata.obs[col] = adata.obs[col].cat.set_categories(sorted(adata.obs[col].cat.categories))
    adata.uns[f"{col}_colors"] = [celltype_color_map[c] for c in adata.obs[col].cat.categories]

fig, axes = plt.subplots(2, 4, figsize=(32, 14), constrained_layout=True)

# Square each panel's plot box BEFORE plotting, so continuous-colorbar panels
# get their colorbar placed right next to the (already square) axes.
for ax in axes.flat:
    ax.set_box_aspect(1)

sc.pl.umap(adata, color=CELLTYPIST_COL,          title="Atlas (input)",             ax=axes[0,0], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="approach1_celltype",     title="Approach 1: scANVI-style",  ax=axes[0,1], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="approach2_celltype",     title="Approach 2: CellTypist",    ax=axes[0,2], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="approach3_celltype",     title="Approach 3: Custom NN",     ax=axes[0,3], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="consensus_celltype",     title="Consensus cell type",       ax=axes[1,0], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="leiden_0.5",             title="Leiden clusters (res=0.5)", ax=axes[1,1], show=False, legend_loc="right margin", legend_fontsize=7)
sc.pl.umap(adata, color="entropy_norm_0.5",       title="Batch entropy per cluster (res=0.5)", ax=axes[1,2], show=False)
axes[1, 3].axis("off")

# scanpy attaches continuous-value colorbars via make_axes_locatable, which
# constrained_layout doesn't manage — snug the colorbar up to the square axes
# by hand, then freeze the layout so a later draw doesn't drift it apart again.
fig.canvas.draw()
cbar_axes = [ax for ax in fig.axes if ax not in axes.flat]
for main_ax, cbar_ax in zip([axes[1, 2]], cbar_axes):
    pos = main_ax.get_position()
    cbar_ax.set_position([pos.x1 + 0.005, pos.y0, 0.012, pos.height])
try:
    fig.set_layout_engine("none")
except AttributeError:
    fig.set_constrained_layout(False)

plt.suptitle("Cell Annotation Evaluation — 3 Approaches + Consensus + Clustering", fontsize=14, y=1.02)
plt.savefig(FINAL_OUT, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FINAL_OUT}")